In [1]:
!pip install fairlearn pgmpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.0/240.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [2]:
import random
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

# ---------------- reproducibility ----------------
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

In [3]:
from sklearn.utils import resample
import pandas as pd

#function to upsample in a proportionally biased way

def upsample_minority_groups_german(data, target_col, group_cols, target_value, total_target_samples):
   

    # matches with target to be upsampled
    target_df = data[data[target_col] == target_value]
    
    #groups by sensitive atrributes
    dist = target_df.groupby(group_cols).size().reset_index(name='count')
    total_current_samples = dist['count'].sum()
    
    
    # print(f"Initial {target_col} == {target_value} samples: {total_current_samples}")
    # print("Initial group distribution:")
    # print(dist)

    #creates a table containing all possible combinations of the sensitive attris in the lower bound label
    #on basis of proportion
    
    # Calculate the upsample count for each group
    dist['up_count'] = (dist['count'] * total_target_samples / total_current_samples).round().astype(int)
    # print("Calculated upsample counts for each group:")
    # print(dist)
    
    # Initialize a list to store the upsampled groups
    upsampled_list = []
    
    for _, row in dist.iterrows():
        # it samples for a specific group
        group_condition = True
        for col in group_cols:
            group_condition = group_condition & (target_df[col] == row[col])
        
        #locating the rows having the current group    
        group_df = target_df.loc[group_condition]
        
        #randomly resampling from the group_df
        upsampled_group = resample(group_df, replace=True, n_samples=row['up_count'], random_state=42)
        upsampled_list.append(upsampled_group)
    
    # Concatenate all upsampled groups
    upsampled_df = pd.concat(upsampled_list)
    
    # Combine the upsampled group with the other class
    other_df = data[data[target_col] != target_value]
    final_df = pd.concat([other_df, upsampled_df]).sample(frac=1, random_state=42).reset_index(drop=True)
    
    # Check the final distribution after upsampling
    # print(f"Final distribution of {target_col} after upsampling:")
    # print(final_df[target_col].value_counts())
    
    return final_df

In [4]:
def preprocess_german_for_fair_bbn(csv_path="/kaggle/input/credit-risk-analysis-dataset/german.data", seed=42):
    import pandas as pd
    import numpy as np
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler, OneHotEncoder
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.utils import resample

    # ---------------- Load Data ----------------
    column_names = [
        "status", "duration", "credit_history", "purpose", "amount", "savings", "employment",
        "installment_rate", "personal_status_sex", "other_debtors", "residence", "property", "age",
        "other_installment_plans", "housing", "number_credits", "job", "people_liable",
        "telephone", "foreign_worker", "credit"
    ]
    
    df = pd.read_csv(csv_path, sep=' ', names=column_names)
    
    # ---------------- Sensitive Attributes ----------------
    sex_map = {'A91':'male','A92':'male','A93':'male','A94':'female','A95':'female'}
    df['sex'] = df['personal_status_sex'].map(sex_map)
    df['sex_label'] = df['sex'].map({'male':0,'female':1})
    
    # Age: 0 = <25, 1 = >=25
    df['age_label'] = (df['age'] >= 25).astype(int)
    
    # Foreign worker: 1 = yes (A201), 0 = no (A202)
    df['foreign_worker_label'] = df['foreign_worker'].map({'A201':1,'A202':0})
    
    # Target: credit = 1 for good credit, 0 for bad
    df['credit_label'] = df['credit'].map({1:1,2:0})
    
    df = df.drop(columns=['personal_status_sex','sex','age','foreign_worker','credit'])
    
    # ---------------- Upsampling minority class ----------------
    def upsample_minority_groups_german(data, target_col, group_cols, target_value, total_target_samples):
        minority = data[data[target_col]==target_value]
        dist = minority.groupby(group_cols).size().reset_index(name='count')
        dist['up_count'] = (dist['count']*total_target_samples/dist['count'].sum()).round().astype(int)
        ups = []
        for _, row in dist.iterrows():
            group = minority
            for col in group_cols:
                group = group[group[col]==row[col]]
            ups.append(resample(group, replace=True, n_samples=row['up_count'], random_state=seed))
        return pd.concat([data[data[target_col]!=target_value]] + ups).sample(frac=1, random_state=seed).reset_index(drop=True)
    
    df = upsample_minority_groups_german(
        data=df,
        target_col='credit_label',
        group_cols=['sex_label','age_label','foreign_worker_label'],
        target_value=0,
        total_target_samples=700
    )
    
    # ---------------- Features ----------------
    X = df.drop(columns=['credit_label'])
    y = df['credit_label'].values
    
    sensitive_sex = df['sex_label'].values
    sensitive_age = df['age_label'].values
    sensitive_foreign = df['foreign_worker_label'].values
    
    numerical_cols = ["duration", "amount", "installment_rate", "residence","number_credits","people_liable"]
    categorical_cols = [col for col in X.columns if col not in numerical_cols]
    
    # Preprocessor for NN
    preproc = ColumnTransformer([
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])
    
    # ---------------- BBN dataframe ----------------
    bbn_df = df.copy()
    for col in numerical_cols:
        bbn_df[col] = pd.qcut(bbn_df[col], 5, labels=False, duplicates='drop').astype(int)
    for col in categorical_cols + ['sex_label','age_label','foreign_worker_label']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    
    # ---------------- Train/Test Split ----------------
    X_train_raw, X_test_raw, y_train, y_test, \
    sens_age_train, sens_age_test, \
    sens_sex_train, sens_sex_test, \
    sens_foreign_train, sens_foreign_test = train_test_split(
        X, y, sensitive_age, sensitive_sex, sensitive_foreign,
        test_size=0.2, random_state=seed, stratify=y
    )
    
    # NN preprocessing
    pipeline = Pipeline([('preprocessor', preproc)])
    X_train_nn = pipeline.fit_transform(X_train_raw)
    X_test_nn = pipeline.transform(X_test_raw)
    
    # BBN datasets
    bbn_train_df = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test_df = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    
    return {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train_df, 'bbn_test_df': bbn_test_df,
        'y_train': y_train, 'y_test': y_test,
        'sens_age_train': sens_age_train, 'sens_age_test': sens_age_test,
        'sens_sex_train': sens_sex_train, 'sens_sex_test': sens_sex_test,
        'sens_foreign_train': sens_foreign_train, 'sens_foreign_test': sens_foreign_test
    }


In [5]:
# ------------------------ Fair BBN for German Dataset ------------------------
class SimpleFairBBN:
    def __init__(self, feature_names, target_name='target'):
        self.feature_names = feature_names
        self.target_name = target_name
        self.model = None
        self.inference = None

    def fit(self, df_discrete, y, include_sensitive=True):
        """
        Fits a Bayesian Belief Network (BBN) on discretized data.
        If include_sensitive=True, includes sensitive attributes: age, sex, and foreign_worker.
        """
        data = df_discrete.copy().reset_index(drop=True)
        data[self.target_name] = y

        # Select core features
        candidate_features = list(df_discrete.columns)
        selected = candidate_features[:6]

        # Include sensitive attributes if requested
        if include_sensitive:
            for sens in ['age_label', 'sex_label', 'foreign_worker_label']:
                if sens in df_discrete.columns and sens not in selected:
                    selected.append(sens)

        # Define directed edges: feature → target
        edges = [(f, self.target_name) for f in selected]

        # Build and fit Bayesian Network
        self.feature_names = selected
        self.model = DiscreteBayesianNetwork(edges)
        self.model.fit(
            data,
            estimator=BayesianEstimator,
            prior_type='BDeu',
            equivalent_sample_size=5
        )
        self.inference = VariableElimination(self.model)

    def predict_proba(self, df_discrete):
        """
        Predicts the probability P(target=1 | evidence) for each row.
        Uses up to 3 observed features as evidence for computational efficiency.
        """
        Xdf = df_discrete.reset_index(drop=True)
        probs = []

        for _, row in Xdf.iterrows():
            evidence = {}
            used = 0
            for f in self.feature_names:
                if f in row and not pd.isna(row[f]) and used < 3:
                    try:
                        evidence[f] = int(row[f])
                        used += 1
                    except:
                        continue

            try:
                if evidence:
                    q = self.inference.query(variables=[self.target_name], evidence=evidence)
                else:
                    q = self.inference.query(variables=[self.target_name])
                p1 = q.values[1] if len(q.values) == 2 else 0.5
            except:
                p1 = 0.5

            probs.append(p1)

        return np.array(probs)


In [6]:
# ---------------- Adapter + Adversary for German Dataset ----------------
import torch
import torch.nn as nn

# Adapter: takes BBN marginals and outputs a binary classification logit
class AdapterNN(nn.Module):
    def __init__(self, in_dim=4, hidden_dim=64):
        """
        in_dim: number of input features from BBN (P(y|all), P(y|age), P(y|sex), P(y|foreign))
        hidden_dim: size of first hidden layer
        """
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.out = nn.Linear(hidden_dim // 2, 1)  # binary classification output

    def forward(self, x, return_features=False):
        h = self.act(self.fc1(x))
        h2 = self.act(self.fc2(h))
        logit = self.out(h2)
        if return_features:
            return logit, h2  # return both prediction logit and hidden representation
        return logit


# Adversary: predicts sensitive attributes (age, sex, foreign) from hidden representation
class AdversaryNN(nn.Module):
    def __init__(self, in_dim, age_classes, sex_classes, foreign_classes):
        """
        in_dim: input dimension from Adapter hidden layer
        age_classes, sex_classes, foreign_classes: number of categories for each sensitive attribute
        """
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(in_dim, 32),
            nn.ReLU()
        )
        self.age_head = nn.Linear(32, age_classes)
        self.sex_head = nn.Linear(32, sex_classes)
        self.foreign_head = nn.Linear(32, foreign_classes)

    def forward(self, x):
        s = self.shared(x)
        return self.age_head(s), self.sex_head(s), self.foreign_head(s)


In [7]:
def train_fair_bbn_adapter_german(data_dict, lambda_adv=0.5, alpha_bbn=0.5, epochs=60, batch_size=256, lr=1e-3, seed=42):
    

    # ---------------- Extract Data ----------------
    X_train_nn = data_dict['X_train_nn']; X_test_nn = data_dict['X_test_nn']
    bbn_train_df = data_dict['bbn_train_df']; bbn_test_df = data_dict['bbn_test_df']
    y_train = data_dict['y_train']; y_test = data_dict['y_test']
    sens_age_train = data_dict['sens_age_train']; sens_age_test = data_dict['sens_age_test']
    sens_sex_train = data_dict['sens_sex_train']; sens_sex_test = data_dict['sens_sex_test']
    sens_foreign_train = data_dict['sens_foreign_train']; sens_foreign_test = data_dict['sens_foreign_test']

    # ---------------- Baseline NN ----------------
    print("Training baseline MLP...")
    baseline = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=seed)
    baseline.fit(X_train_nn, y_train)
    base_pred = baseline.predict(X_test_nn)
    base_acc = accuracy_score(y_test, base_pred)
    
    # Evaluate fairness wrt sensitive attributes
    base_sex_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_sex_test)
    base_sex_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_sex_test)
    base_age_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_age_test)
    base_age_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_age_test)
    base_foreign_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_foreign_test)
    base_foreign_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_foreign_test)

    print(f"Baseline MLP Accuracy: {base_acc:.4f}")

    # ---------------- Fair BBN ----------------
    print("Training Fair BBN...")
    bbn = SimpleFairBBN(feature_names=list(bbn_train_df.columns))
    bbn.fit(bbn_train_df, y_train, include_sensitive=True)

    # Compute conditional probabilities as adapter input
    p_all = bbn.predict_proba(bbn_train_df).reshape(-1,1)
    p_age = bbn.predict_proba(bbn_train_df[['age_label']]).reshape(-1,1)
    p_sex = bbn.predict_proba(bbn_train_df[['sex_label']]).reshape(-1,1)
    p_foreign = bbn.predict_proba(bbn_train_df[['foreign_worker_label']]).reshape(-1,1)
    adapter_in_train = torch.tensor(np.hstack([p_all, p_age, p_sex, p_foreign]), dtype=torch.float32)

    p_all_test = bbn.predict_proba(bbn_test_df).reshape(-1,1)
    p_age_test = bbn.predict_proba(bbn_test_df[['age_label']]).reshape(-1,1)
    p_sex_test = bbn.predict_proba(bbn_test_df[['sex_label']]).reshape(-1,1)
    p_foreign_test = bbn.predict_proba(bbn_test_df[['foreign_worker_label']]).reshape(-1,1)
    adapter_in_test = torch.tensor(np.hstack([p_all_test, p_age_test, p_sex_test, p_foreign_test]), dtype=torch.float32)

    # Convert target + sensitive features to tensors
    y_train_t = torch.tensor(y_train.reshape(-1,1), dtype=torch.float32)
    y_test_t  = torch.tensor(y_test.reshape(-1,1), dtype=torch.float32)
    age_train_t = torch.tensor(sens_age_train.astype(int), dtype=torch.long)
    sex_train_t = torch.tensor(sens_sex_train.astype(int), dtype=torch.long)
    foreign_train_t = torch.tensor(sens_foreign_train.astype(int), dtype=torch.long)

    # Dataloader for mini-batch training
    train_loader = DataLoader(
        TensorDataset(adapter_in_train, y_train_t, age_train_t, sex_train_t, foreign_train_t),
        batch_size=batch_size, shuffle=True
    )

    # ---------------- Define Networks ----------------
    adapter = AdapterNN(in_dim=4, hidden_dim=64)
   
    adversary = AdversaryNN(
    in_dim=32,
    age_classes=2,
    sex_classes=2,
    foreign_classes=2
)

    adapter_opt = optim.Adam(adapter.parameters(), lr=lr)
    adv_opt = optim.Adam(adversary.parameters(), lr=lr)
    pred_loss_fn = nn.BCEWithLogitsLoss()
    adv_loss_fn = nn.CrossEntropyLoss()

    # ---------------- Training Loop ----------------
    print("Training Adapter with Adversarial + Fairness Regularization...")
    for epoch in range(epochs):
        adapter.train(); adversary.train()
        total_adapter_loss = 0.0; total_adv_loss = 0.0

        for batch in train_loader:
            batch_in, batch_y, batch_age, batch_sex, batch_foreign = batch

            # --- Train Adversary ---
            with torch.no_grad():
                _, features = adapter(batch_in, return_features=True)
            adv_opt.zero_grad()
            age_logits, sex_logits, foreign_logits = adversary(features)
            adv_loss = (
                adv_loss_fn(age_logits, batch_age) +
                adv_loss_fn(sex_logits, batch_sex) +
                adv_loss_fn(foreign_logits, batch_foreign)
            ) / 3.0
            adv_loss.backward(); adv_opt.step()
            total_adv_loss += adv_loss.item()

            # --- Train Adapter ---
            adapter_opt.zero_grad()
            logit, features = adapter(batch_in, return_features=True)
            pred_loss = pred_loss_fn(logit, batch_y)
            age_logits2, sex_logits2, foreign_logits2 = adversary(features)
            adv_loss_for_adapter = (
                adv_loss_fn(age_logits2, batch_age) +
                adv_loss_fn(sex_logits2, batch_sex) +
                adv_loss_fn(foreign_logits2, batch_foreign)
            ) / 3.0

            # Fairness penalty (simple DP)
            dp_penalty = torch.abs(features[batch_sex==0].mean() - features[batch_sex==1].mean())
            total_loss = pred_loss - lambda_adv * adv_loss_for_adapter + alpha_bbn * dp_penalty
            total_loss.backward()
            adapter_opt.step()
            total_adapter_loss += total_loss.item()

        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch:3d} | Adv Loss: {total_adv_loss/len(train_loader):.4f} | Adapter Loss: {total_adapter_loss/len(train_loader):.4f}")

    # ---------------- Evaluation ----------------
    adapter.eval(); adversary.eval()
    with torch.no_grad():
        test_logit,_ = adapter(adapter_in_test, return_features=True)
        test_probs = torch.sigmoid(test_logit.cpu()).numpy().flatten()
        test_pred = (test_probs > 0.5).astype(int)

    adv_acc = accuracy_score(y_test, test_pred)
    adv_sex_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_sex_test)
    adv_sex_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_sex_test)
    adv_age_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_age_test)
    adv_age_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_age_test)
    adv_foreign_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_foreign_test)
    adv_foreign_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_foreign_test)

    # ---------------- Results ----------------
    print("\nBASELINE MLP RESULTS:")
    print(f" Accuracy: {base_acc:.4f}")
    print(f" Sex DP: {base_sex_dp:.4f}, Sex EO: {base_sex_eo:.4f}")
    print(f" Age DP: {base_age_dp:.4f}, Age EO: {base_age_eo:.4f}")
    print(f" Foreign DP: {base_foreign_dp:.4f}, Foreign EO: {base_foreign_eo:.4f}")

    print("\nBBN + Adapter (Adversarial + Fairness) RESULTS:")
    print(f" Accuracy: {adv_acc:.4f}")
    print(f" Sex DP: {adv_sex_dp:.4f}, Sex EO: {adv_sex_eo:.4f}")
    print(f" Age DP: {adv_age_dp:.4f}, Age EO: {adv_age_eo:.4f}")
    print(f" Foreign DP: {adv_foreign_dp:.4f}, Foreign EO: {adv_foreign_eo:.4f}")

    return {
        'baseline': {
            'pred': base_pred, 'acc': base_acc,
            'sex_dp': base_sex_dp, 'sex_eo': base_sex_eo,
            'age_dp': base_age_dp, 'age_eo': base_age_eo,
            'foreign_dp': base_foreign_dp, 'foreign_eo': base_foreign_eo
        },
        'bbn_adapter': {
            'pred': test_pred, 'acc': adv_acc,
            'sex_dp': adv_sex_dp, 'sex_eo': adv_sex_eo,
            'age_dp': adv_age_dp, 'age_eo': adv_age_eo,
            'foreign_dp': adv_foreign_dp, 'foreign_eo': adv_foreign_eo
        }
    }


In [8]:
# ---------------- Run ----------------
if __name__=='__main__':
    print("Preprocessing dataset...")
    data_dict = preprocess_german_for_fair_bbn()
    results = train_fair_bbn_adapter_german(data_dict, lambda_adv=0.5, alpha_bbn=0.5, epochs=60)


Preprocessing dataset...
Training baseline MLP...
Baseline MLP Accuracy: 0.8571
Training Fair BBN...
Training Adapter with Adversarial + Fairness Regularization...
Epoch   0 | Adv Loss: 0.6125 | Adapter Loss: 0.3911
Epoch  10 | Adv Loss: 0.4827 | Adapter Loss: 0.4563
Epoch  20 | Adv Loss: 0.3877 | Adapter Loss: 0.5008
Epoch  30 | Adv Loss: 0.3191 | Adapter Loss: 0.5341
Epoch  40 | Adv Loss: 0.3024 | Adapter Loss: 0.5425
Epoch  50 | Adv Loss: 0.2959 | Adapter Loss: 0.5453
Epoch  59 | Adv Loss: 0.2831 | Adapter Loss: 0.5516

BASELINE MLP RESULTS:
 Accuracy: 0.8571
 Sex DP: 0.0461, Sex EO: 0.0945
 Age DP: 0.0940, Age EO: 0.0731
 Foreign DP: 0.4361, Foreign EO: 0.2188

BBN + Adapter (Adversarial + Fairness) RESULTS:
 Accuracy: 0.5000
 Sex DP: 0.0000, Sex EO: 0.0000
 Age DP: 0.0000, Age EO: 0.0000
 Foreign DP: 0.0000, Foreign EO: 0.0000
